# Loss-Grid Computer: Functional Eval T4 Suite

Runs the functional-evaluation redesign benchmark on Colab and saves a complete run bundle to Google Drive.

This notebook automates:
- Drive mount and persistent artifact storage
- repo loading from Drive or existing Colab workspace
- asset linking from Drive into the repo
- API probe capture via `src.functional_eval.api_pipeline`
- benchmark execution via `src.functional_eval.experiment`
- Drive export of probe output, benchmark summary, and run manifest

Target runtime is a CUDA T4. The notebook fails fast on non-T4 runtimes because the PRD treats T4 as the definitive comparison backend.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Root folder in Drive where assets live and benchmark bundles will be saved.
# Expected layout:
#   DRIVE_ROOT/
#     assets/
#       cifar-10-batches-py/
#       cifar10-resnet20-0.pkl
#     functional_eval_runs/
#     loss-grid-computer/
DRIVE_ROOT = '/content/drive/MyDrive/loss-grid-experiments'
DRIVE_ASSETS_ROOT = f'{DRIVE_ROOT}/assets'
DRIVE_RESULTS_ROOT = f'{DRIVE_ROOT}/functional_eval_runs'

## 2. Load repo and install dependencies

In [ ]:
import os
from pathlib import Path

if 'DRIVE_ROOT' not in globals():
    DRIVE_ROOT = '/content/drive/MyDrive/loss-grid-experiments'
    DRIVE_ASSETS_ROOT = f'{DRIVE_ROOT}/assets'
    DRIVE_RESULTS_ROOT = f'{DRIVE_ROOT}/functional_eval_runs'

REPO_DIR = Path('/content/loss-grid-computer')
DRIVE_REPO_DIR = Path(DRIVE_ROOT) / 'loss-grid-computer'

if REPO_DIR.exists():
    print(f'Using existing workspace repo: {REPO_DIR}')
elif DRIVE_REPO_DIR.exists():
    REPO_DIR.symlink_to(DRIVE_REPO_DIR, target_is_directory=True)
    print(f'Linked repo from Drive: {REPO_DIR} -> {DRIVE_REPO_DIR}')
else:
    raise FileNotFoundError(
        f'Repo not found at {REPO_DIR} or {DRIVE_REPO_DIR}. '
        f'Place the project folder at {DRIVE_REPO_DIR} and rerun.'
    )

os.chdir(REPO_DIR)
print(f'Working directory: {Path.cwd()}')

In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'torch',
    'torchvision',
    'scikit-learn',
    'pandas',
    'numpy',
    'matplotlib',
    'sympy>=1.13,<1.14',
])
print('Dependencies installed.')

## 3. Link assets from Drive

In [ ]:
import os
from pathlib import Path

assets_dst = Path('assets')
assets_src = Path(DRIVE_ASSETS_ROOT)

if assets_dst.exists() and not assets_dst.is_symlink():
    print(f'Assets already present in repo: {assets_dst.resolve()}')
elif assets_dst.is_symlink():
    print(f'Assets symlink already present: {assets_dst} -> {os.readlink(assets_dst)}')
elif assets_src.exists():
    assets_dst.symlink_to(assets_src)
    print(f'Linked assets from Drive: {assets_dst} -> {assets_src}')
else:
    assets_dst.mkdir(exist_ok=True)
    print(f'WARNING: {assets_src} not found. Create it in Drive and add the required files.')

required = [
    'assets/cifar-10-batches-py',
    'assets/cifar10-resnet20-0.pkl',
]
missing = [path for path in required if not Path(path).exists()]
if missing:
    raise FileNotFoundError('Missing required assets:\n' + '\n'.join(missing))

print('Required assets are present.')

## 4. Verify T4 runtime

In [ ]:
import os
import torch

assert torch.cuda.is_available(), 'No CUDA GPU found. Change Runtime > Change runtime type > T4 GPU.'
gpu_name = torch.cuda.get_device_name(0)
assert 't4' in gpu_name.lower(), f'Expected a T4 runtime, found {gpu_name!r}'
props = torch.cuda.get_device_properties(0)
print(f'GPU: {gpu_name}')
print(f'VRAM: {props.total_memory / 1e9:.1f} GB')
print(f'CPU cores: {os.cpu_count()}')
print(f'Torch: {torch.__version__}')

## 5. Experiment configuration

These defaults match the current PRD target for definitive comparison: CIFAR-10, ResNet20, `8x8` grid, `3` repeats, and chunk-size exploration for `vmap`.

In [ ]:
SEED = 1337
DEVICE = 'cuda'
GRID_RESOLUTION = 8
GRID_SCALE = 1.0
GPU_BATCH_SIZE = 32
SAMPLE_COUNT = 1024
REPEATS = 3
POINT_CHUNK_SIZES = (1, 2, 4, 8, 16, 32, 64)
MAX_MEMORY_FRACTION = 0.85

print({
    'seed': SEED,
    'device': DEVICE,
    'grid_resolution': GRID_RESOLUTION,
    'grid_scale': GRID_SCALE,
    'gpu_batch_size': GPU_BATCH_SIZE,
    'sample_count': SAMPLE_COUNT,
    'repeats': REPEATS,
    'point_chunk_sizes': POINT_CHUNK_SIZES,
    'max_memory_fraction': MAX_MEMORY_FRACTION,
})

## 6. Run API probe and benchmark suite

In [ ]:
import importlib
import json
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

repo_candidates = [Path.cwd(), Path.cwd().parent, REPO_DIR]
repo_root = next((path.resolve() for path in repo_candidates if (path / 'src').is_dir()), None)
if repo_root is None:
    checked = ', '.join(str(path.resolve()) for path in repo_candidates)
    raise ModuleNotFoundError(f"Could not locate repo root containing src/. Checked: {checked}")

functional_eval_dir = repo_root / 'src' / 'functional_eval'
if not functional_eval_dir.is_dir():
    raise ModuleNotFoundError(
        "This checkout does not include src/functional_eval required by this notebook. "
        f"repo_root={repo_root}. "
        "Sync the repository to a revision containing src/functional_eval and rerun from the top."
    )

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

for mod in list(sys.modules):
    if mod.startswith('src.functional_eval') or mod in {'src', 'src.system_schema', 'src.workloads'}:
        del sys.modules[mod]

src_module = importlib.import_module('src')
src_file = Path(src_module.__file__).resolve()
if not src_file.is_relative_to(repo_root):
    raise ModuleNotFoundError(f"Imported 'src' from {src_file}, expected under {repo_root}")

api_module = importlib.import_module('src.functional_eval.api_pipeline')
experiment_module = importlib.import_module('src.functional_eval.experiment')

api_result = api_module.run_pipeline(DEVICE, seed=SEED)
print(json.dumps(api_result, indent=2, sort_keys=True))

request = experiment_module.build_default_request(
    device=DEVICE,
    sample_count=SAMPLE_COUNT,
    batch_size=GPU_BATCH_SIZE,
    resolution=GRID_RESOLUTION,
    scale=GRID_SCALE,
)
config = experiment_module.FunctionalEvalConfig(
    request=request,
    seed=SEED,
    repeats=REPEATS,
    point_chunk_sizes=POINT_CHUNK_SIZES,
    max_memory_fraction=MAX_MEMORY_FRACTION,
)
summary = experiment_module.run_experiment(config)

timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
run_dir = Path(DRIVE_RESULTS_ROOT) / timestamp
run_dir.mkdir(parents=True, exist_ok=True)

api_path = run_dir / 'api-pipeline.json'
api_path.write_text(json.dumps(api_result, indent=2, sort_keys=True), encoding='utf-8')

local_summary_path = Path(summary['output_path'])
drive_summary_path = run_dir / 'experiment-summary.json'
shutil.copy2(local_summary_path, drive_summary_path)

manifest = {
    'created_at': timestamp,
    'repo_root': str(repo_root),
    'drive_root': DRIVE_ROOT,
    'run_dir': str(run_dir),
    'api_pipeline_path': str(api_path),
    'local_experiment_summary_path': str(local_summary_path),
    'drive_experiment_summary_path': str(drive_summary_path),
    'config': {
        'seed': SEED,
        'device': DEVICE,
        'grid_resolution': GRID_RESOLUTION,
        'grid_scale': GRID_SCALE,
        'gpu_batch_size': GPU_BATCH_SIZE,
        'sample_count': SAMPLE_COUNT,
        'repeats': REPEATS,
        'point_chunk_sizes': list(POINT_CHUNK_SIZES),
        'max_memory_fraction': MAX_MEMORY_FRACTION,
    },
    'platform': summary['platform'],
    'candidate_summary': summary['candidate_summary'],
}
manifest_path = run_dir / 'run-manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding='utf-8')

print(f'Run bundle saved to {run_dir}')
print(f'API probe: {api_path}')
print(f'Experiment summary: {drive_summary_path}')
print(f'Manifest: {manifest_path}')

## 7. Candidate summary

In [ ]:
import pandas as pd

candidate_df = pd.DataFrame(summary['candidate_summary'])
candidate_df = candidate_df.sort_values(['all_validations_passed', 'mean_speedup_vs_baseline'], ascending=[False, False], na_position='last')
candidate_df

## 8. Next-step view

This view isolates the numerically valid candidates and orders them by mean speedup versus the original baseline.

In [ ]:
valid_df = candidate_df[candidate_df['all_validations_passed'] == True].copy()
valid_df = valid_df.sort_values('mean_speedup_vs_baseline', ascending=False)
valid_df[['candidate', 'mean_total_grid_s', 'mean_speedup_vs_baseline', 'status_counts']]